# 01 · Data Cleaning

Loads the raw social media dataset, checks for quality issues, and produces
a cleaned CSV that the rest of the notebooks build on. Keeping this as a
single source of truth avoids re-cleaning the data differently in every
notebook (and re-reading a hardcoded path that only exists on one machine).

In [1]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/Social_Media_Sentiment_Analysis.csv"
CLEAN_PATH = "../data/cleaned_social_media_sentiment.csv"

df = pd.read_csv(RAW_PATH)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.info()

Loaded 2,200 rows, 27 columns
<class 'pandas.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   platform              2200 non-null   str    
 1   post_id               2200 non-null   str    
 2   user_id               2200 non-null   str    
 3   username              2200 non-null   str    
 4   user_verified         2200 non-null   bool   
 5   user_followers_count  2200 non-null   int64  
 6   user_location         2200 non-null   str    
 7   post_text             2200 non-null   str    
 8   language              2200 non-null   str    
 9   hashtags              2200 non-null   str    
 10  mentions              872 non-null    str    
 11  post_length           2200 non-null   int64  
 12  like_count            2200 non-null   int64  
 13  comment_count         2200 non-null   int64  
 14  share_count           2200 non-null   int64  
 15  en

## Missing values

Check what's actually missing before deciding how to handle it.

In [2]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

mentions    1328
dtype: int64

## Cleaning steps

- Drop exact duplicate rows
- Normalize column names (lowercase, stripped, underscored)
- Strip whitespace from text columns
- Fill numeric gaps with the column median, categorical gaps with the mode
- Parse `posted_datetime` into an actual datetime type

In [3]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

for col in df.select_dtypes(include="object"):
    df[col] = df[col].str.strip()

df = df.replace("", np.nan)

for col in df.select_dtypes(include=["int64", "float64"]):
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object"):
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

if "posted_datetime" in df.columns:
    df["posted_datetime"] = pd.to_datetime(df["posted_datetime"], errors="coerce")

df = df.dropna(how="all").reset_index(drop=True)

print(f"Final shape: {df.shape}")
df.head()

Dropped 0 duplicate rows
Final shape: (2200, 27)


/tmp/ipykernel_689/3410117533.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object"):
/tmp/ipykernel_689/3410117533.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details o

,platform,post_id,user_id,username,user_verified,user_followers_count,user_location,post_text,language,hashtags,...,day_of_week,is_trending_topic,topic_category,sentiment_label,sentiment_score,emotion_label,toxicity_score,sarcasm_detected,spam_flag,data_source_url
0,YouTube,YO_100000,user_2679,rxckafna,True,375410,Germany,I don’t agree with this at all.,ur,"#news,#update",...,Wed,True,Technology,Positive,0.82,Sad,0.10,False,True,https://www.youtube.com
1,Reddit,RE_100001,user_3045,zjovqwps,True,346702,UK,Worst decision ever made.,fr,"#trending,#ai",...,Tue,False,Sports,Neutral,-0.02,Angry,0.67,True,True,https://www.reddit.com
2,YouTube,YO_100002,user_4598,rvufaigf,False,111487,Germany,This trend is getting out of hand.,de,"#news,#update",...,Fri,False,Finance,Positive,0.62,Neutral,0.43,False,False,https://www.youtube.com
3,Twitter,TW_100003,user_3504,qukbjznz,True,356674,Germany,Pretty neutral about the update.,en,"#trending,#ai",...,Thu,True,Finance,Neutral,0.14,Sad,0.97,True,True,https://www.twitter.com
4,YouTube,YO_100004,user_3646,ounaiayw,False,125551,USA,"Loved the explanation, very helpful.",es,"#trending,#ai",...,Sun,True,Technology,Neutral,-0.05,Fear,0.95,True,False,https://www.youtube.com


## Sanity check

Confirm nothing is left null and the dtypes look right before saving.

In [4]:
assert df.isnull().sum().sum() == 0, "Unexpected nulls remain"
df.dtypes

platform                           str
post_id                            str
user_id                            str
username                           str
user_verified                     bool
user_followers_count             int64
user_location                      str
post_text                          str
language                           str
hashtags                           str
mentions                           str
post_length                      int64
like_count                       int64
comment_count                    int64
share_count                      int64
engagement_score               float64
posted_datetime         datetime64[us]
day_of_week                        str
is_trending_topic                 bool
topic_category                     str
sentiment_label                    str
sentiment_score                float64
emotion_label                      str
toxicity_score                 float64
sarcasm_detected                  bool
spam_flag                

In [5]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved cleaned dataset to {CLEAN_PATH}")

Saved cleaned dataset to ../data/cleaned_social_media_sentiment.csv
